# Personal Intelligence Engine - Phase 2 OCR (Kaggle + Qwen2-VL-7B, T4 x2)

**Setup for Background Execution:**
1. **Add Data**: Upload your `notebooks_pdf` folder as a Kaggle Dataset, then click "Add Input" to attach it.
2. **Settings (right sidebar)**:
   - Accelerator: **GPU T4 x2** (2 × 16 GB VRAM — each GPU runs its own model instance)
   - Internet: **On** (needed to download model weights)
3. **Run**: Click **"Save Version"** → **"Save & Run All (Commit)"**
4. **Download**: Go to Output tab when done → download `output.zip`

In [ ]:
# Cell 1: Install Dependencies
import subprocess, sys

pkgs = [
    "transformers>=4.45.0",
    "bitsandbytes>=0.43.0",
    "accelerate>=0.30.0",
    "qwen-vl-utils",
    "pypdfium2",
    "Pillow",
    "tqdm",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)

import os

INPUT_ROOT = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working/notebooks_md"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Dependencies installed.")

In [ ]:
# Cell 2: Find Input Data
import os

print("Searching for notebooks_pdf in /kaggle/input...")

pdf_root = None
# Walk input directory to find the "notebooks_pdf" folder or root of pdfs
for root, dirs, files in os.walk(INPUT_ROOT):
    if "notebooks_pdf" in dirs:
        pdf_root = os.path.join(root, "notebooks_pdf")
        break
    # Fallback: looks for a folder that contains _structure.json files
    for f in files:
        if f == "_structure.json":
            pdf_root = os.path.dirname(root) # Parent of the notebook folder
            break
    if pdf_root:
        break

if not pdf_root:
    # If user uploaded contents directly to dataset root
    pdf_root = INPUT_ROOT

print(f"✅ PDF Root detected at: {pdf_root}")

# List potential notebooks
notebook_folders = []
for item in os.listdir(pdf_root):
    path = os.path.join(pdf_root, item)
    if os.path.isdir(path):
        # Check if it looks like a notebook folder (has _structure.json or PDFs)
        has_structure = os.path.exists(os.path.join(path, "_structure.json"))
        has_pdfs = any(f.endswith('.pdf') for f in os.listdir(path))
        
        if has_structure or has_pdfs:
            notebook_folders.append(path)

print(f"Found {len(notebook_folders)} notebook folder(s)")

In [ ]:
# Cell 2.5: Fix Windows Paths in Structure Files
import json
import os

def windows_basename(path):
    """Extract filename from a Windows or Unix path, works on either OS."""
    # Replace backslashes so split('/') works universally
    return path.replace("\\", "/").split("/")[-1]

print("=" * 60)
print("Checking and fixing paths in structure files...")
print("=" * 60)

fixed_structures = {}

for nb_path in notebook_folders:
    nb_folder_name = os.path.basename(nb_path)
    structure_file = os.path.join(nb_path, "_structure.json")

    if not os.path.exists(structure_file):
        print(f"\n⚠ No _structure.json found in: {nb_folder_name}")
        continue

    with open(structure_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    pages = data.get("pages", [])
    print(f"\n📚 {nb_folder_name}: {len(pages)} page(s) in structure")

    if not pages:
        print("  ⚠ Structure file is empty!")
        continue

    # Check if paths need fixing (Windows paths contain backslashes or drive letters)
    first_path = pages[0].get("path", "")
    needs_fix = "\\" in first_path or (len(first_path) > 1 and first_path[1] == ":")

    if not needs_fix:
        existing = sum(1 for p in pages if os.path.exists(p.get("path", "")))
        print(f"  ✓ Paths look correct ({existing}/{len(pages)} files exist)")
        fixed_structures[nb_path] = structure_file
        continue

    print(f"  ⚠ Windows paths detected — remapping to Kaggle paths...")

    # Build filename → kaggle_path lookup by scanning the notebook folder
    filename_map = {}
    for root, _, files in os.walk(nb_path):
        for file in files:
            if file.endswith(".pdf"):
                filename_map[file] = os.path.join(root, file)

    print(f"  Found {len(filename_map)} PDF(s) on disk")

    # Debug: show a sample of what we're trying to match
    sample_stored = windows_basename(pages[0].get("path", ""))
    sample_disk = next(iter(filename_map), "(none)")
    print(f"  Sample stored filename : {sample_stored}")
    print(f"  Sample disk filename   : {sample_disk}")

    fixed = 0
    missing = 0
    missing_names = []
    for p in pages:
        # Use windows_basename so backslash paths parse correctly on Linux
        filename = windows_basename(p.get("path", ""))
        if filename in filename_map:
            p["path"] = filename_map[filename]
            fixed += 1
        else:
            missing += 1
            if len(missing_names) < 3:
                missing_names.append(filename)

    if missing_names:
        print(f"  First missing filename(s): {missing_names}")

    # Save fixed structure to writable /kaggle/working
    fixed_structure_path = os.path.join(
        "/kaggle/working", f"{nb_folder_name}_structure_fixed.json"
    )
    with open(fixed_structure_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

    fixed_structures[nb_path] = fixed_structure_path

    print(f"  ✓ Fixed {fixed}/{len(pages)} path(s) → {fixed_structure_path}")
    if missing:
        print(f"  ⚠ {missing} PDF(s) could not be matched by filename")

print(f"\n{'='*60}")
print("✅ Path check complete")
print("=" * 60)

In [ ]:
# Cell 3: Check GPUs and write worker script
# ─────────────────────────────────────────────────────────────────────────────
# WHY SUBPROCESSES?
#   bitsandbytes initialises CUDA kernels on *all* visible GPUs the moment the
#   first model loads, consuming ~14 GB on GPU 1 before we can load the second
#   model instance there.  The only reliable fix is to spawn one *process* per
#   GPU with CUDA_VISIBLE_DEVICES set in its environment *before* Python starts,
#   so each process sees exactly one GPU.
# ─────────────────────────────────────────────────────────────────────────────
import os, textwrap, torch

num_gpus = torch.cuda.device_count()
print(f"GPUs available: {num_gpus}")
for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({props.total_memory / 1e9:.1f} GB)")

WORKER_SCRIPT = "/kaggle/working/ocr_worker.py"

# ── Worker code ──────────────────────────────────────────────────────────────
# Written as a regular indented block; textwrap.dedent strips the 4-space
# prefix so the resulting file is valid top-level Python.
# IMPORTANT: no backslash escape sequences appear inside the triple-quoted
# string so embedding is safe.  Newlines are produced with chr(10) at runtime.
worker_code = textwrap.dedent("""
    import os, sys, json, time

    # CUDA_VISIBLE_DEVICES is set by the parent process before this image
    # starts, so torch/bitsandbytes only ever see the one assigned GPU.
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    import torch
    from transformers import (
        Qwen2VLForConditionalGeneration,
        AutoProcessor,
        BitsAndBytesConfig,
    )
    import pypdfium2 as pdfium
    from qwen_vl_utils import process_vision_info

    MODEL_ID       = sys.argv[1]
    PAGES_FILE     = sys.argv[2]
    OUTPUT_DIR     = sys.argv[3]
    LOG_FILE       = sys.argv[4]
    DPI            = int(sys.argv[5])
    MAX_NEW_TOKENS = int(sys.argv[6])
    WORKER_ID      = sys.argv[7]

    TRANSCRIPTION_PROMPT = (
        "Transcribe all content from this image exactly as written. "
        "Use LaTeX notation for all mathematical expressions: "
        "inline math wrapped in $...$ and display/block math wrapped in $$...$$. "
        "For any diagrams, drawings, graphs, or figures, provide a concise "
        "description in [square brackets]. "
        "Preserve headings, bullet points, numbered lists, and other structural "
        "elements as markdown. "
        "Do not interpret, summarize, or add any content not present in the image. "
        "Do not add any preamble or closing remarks."
    )

    NL = chr(10)    # newline — avoids backslash sequences inside the template

    def log(msg):
        ts   = time.strftime("%H:%M:%S")
        line = ts + " [" + WORKER_ID + "] " + msg
        with open(LOG_FILE, "a") as lf:
            lf.write(line + NL)
        print(line, flush=True)

    def safe_filename(notebook, section, page_name, order):
        def sanitize(s):
            return "".join(c if c.isalnum() or c in " -_" else "_" for c in s).strip()
        sec = section.replace("/", "_")
        return (
            sanitize(notebook)[:50] + "__"
            + sanitize(sec)[:50] + "__"
            + str(order).zfill(3) + "_"
            + sanitize(page_name)[:50] + ".md"
        )

    def fmt(s):
        if s < 60:   return str(round(s))           + "s"
        if s < 3600: return str(round(s / 60,   1)) + "m"
        return               str(round(s / 3600, 1)) + "h"

    log("Starting on " + torch.cuda.get_device_name(0))

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb,
        device_map="cuda:0",
        trust_remote_code=True,
    )
    model.eval()
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=256 * 28 * 28,
        max_pixels=1280 * 28 * 28,
        trust_remote_code=True,
    )
    log("Model loaded. VRAM: " + str(round(torch.cuda.memory_allocated(0) / 1e9, 2)) + " GB")

    with open(PAGES_FILE) as pf:
        pages = json.load(pf)
    log("Processing " + str(len(pages)) + " page(s)")

    t0   = time.time()
    done = 0

    for idx, page in enumerate(pages):
        pdf_path      = page["path"]
        section       = page.get("section",  "Uncategorized")
        page_name     = page["name"]
        order         = page.get("order",    0)
        notebook_name = page.get("notebook", "Notebook")

        md_filename = safe_filename(notebook_name, section, page_name, order)
        md_path     = os.path.join(OUTPUT_DIR, md_filename)

        if os.path.exists(md_path):
            log("SKIP [" + str(idx + 1) + "/" + str(len(pages)) + "] " + page_name)
            continue

        ps = time.time()
        try:
            doc  = pdfium.PdfDocument(pdf_path)
            scale = DPI / 72.0
            pdf_images = []
            for pi in range(len(doc)):
                pg = doc[pi]
                pdf_images.append(pg.render(scale=scale, rotation=0).to_pil())
                pg.close()
            doc.close()

            parts = []
            for img in pdf_images:
                msgs = [{
                    "role": "user",
                    "content": [
                        {"type": "image", "image": img},
                        {"type": "text",  "text":  TRANSCRIPTION_PROMPT},
                    ],
                }]
                txt    = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
                img_in, _ = process_vision_info(msgs)
                inp    = processor(text=[txt], images=img_in, padding=True, return_tensors="pt").to("cuda:0")
                with torch.no_grad():
                    out_ids = model.generate(**inp, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
                gen = [out_ids[j][len(inp.input_ids[j]):] for j in range(len(out_ids))]
                parts.append(
                    processor.batch_decode(
                        gen,
                        skip_special_tokens=True,
                        clean_up_tokenization_spaces=False,
                    )[0].strip()
                )
                torch.cuda.empty_cache()

            header = NL.join([
                "---",
                "source_pdf: " + pdf_path,
                "notebook: "   + notebook_name,
                "section: "    + section,
                "page: "       + page_name,
                "order: "      + str(order),
                "---",
                "",
                "# " + page_name,
                "",
            ]) + NL

            with open(md_path, "w", encoding="utf-8") as mf:
                mf.write(header + (NL + NL).join(parts))

            done += 1
            ep  = time.time() - ps
            avg = (time.time() - t0) / done
            eta = avg * (len(pages) - (idx + 1))
            log(
                "OK ["  + str(idx + 1) + "/" + str(len(pages)) + "] " + page_name
                + " ("  + str(round(ep,  1)) + "s"
                + ", avg " + str(round(avg, 1)) + "s"
                + ", ETA " + fmt(eta) + ")"
            )

        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            log("OOM [" + str(idx + 1) + "/" + str(len(pages)) + "] " + page_name + ", skipping")
        except Exception as e:
            log("ERR [" + str(idx + 1) + "/" + str(len(pages)) + "] " + page_name + ": " + str(e))

    log("Done. " + str(done) + "/" + str(len(pages)) + " pages in " + fmt(time.time() - t0))
""").lstrip()

with open(WORKER_SCRIPT, "w") as f:
    f.write(worker_code)

print(f"Worker script written to: {WORKER_SCRIPT}")
print(f"Will launch {num_gpus} subprocess(es), one per GPU.")


In [ ]:
# Cell 4: Build page lists, launch one subprocess per GPU, stream their output
import json, os, sys, time, subprocess, threading

# ── Config ───────────────────────────────────────────────────────────────────
MODEL_ID       = "Qwen/Qwen2-VL-7B-Instruct"
DPI            = 200
MAX_NEW_TOKENS = 2500
# ─────────────────────────────────────────────────────────────────────────────

def fmt(s):
    if s < 60:   return f"{s:.0f}s"
    if s < 3600: return f"{s/60:.1f}m"
    return f"{s/3600:.1f}h"

_fixed = fixed_structures if "fixed_structures" in dir() else {}

print("=" * 60)
print("Phase 2: PDF → Qwen2-VL-7B → Markdown")
print(f"         {num_gpus} GPU worker(s), {DPI} DPI, {MAX_NEW_TOKENS} tok/page")
print("=" * 60)
sys.stdout.flush()

overall_start = time.time()

for nb_path in notebook_folders:
    nb_folder_name = os.path.basename(nb_path)
    print(f"\nNotebook: {nb_folder_name}")
    sys.stdout.flush()

    # ── Load page list ────────────────────────────────────────────────────────
    pages         = []
    notebook_name = nb_folder_name
    structure_file = _fixed.get(nb_path, os.path.join(nb_path, "_structure.json"))

    if os.path.exists(structure_file):
        with open(structure_file, "r", encoding="utf-8") as f:
            data = json.load(f)
        notebook_name = data.get("notebook", nb_folder_name)
        pages = [p for p in data.get("pages", []) if os.path.exists(p.get("path", ""))]
        for p in pages:
            p.setdefault("notebook", notebook_name)
        print(f"  {len(pages)} valid page(s) from structure file")
    else:
        for root, _, files in os.walk(nb_path):
            for i, fname in enumerate(sorted(files)):
                if fname.endswith(".pdf"):
                    pages.append({
                        "path":     os.path.join(root, fname),
                        "name":     os.path.splitext(fname)[0],
                        "section":  os.path.relpath(root, nb_path),
                        "order":    i,
                        "notebook": notebook_name,
                    })
        print(f"  {len(pages)} PDFs scanned from directory")

    if not pages:
        print("  ⚠ No pages found — skipping.")
        sys.stdout.flush()
        continue

    # ── Split pages evenly across GPUs ───────────────────────────────────────
    chunks = [pages[i::num_gpus] for i in range(num_gpus)]
    print(f"  Splitting: {[len(c) for c in chunks]} pages per GPU")
    sys.stdout.flush()

    procs     = []   # (gpu_id, Popen)
    log_files = []   # (gpu_id, path)

    for gpu_id, chunk in enumerate(chunks):
        if not chunk:
            continue

        pages_file = f"/kaggle/working/pages_gpu{gpu_id}.json"
        log_file   = f"/kaggle/working/worker_gpu{gpu_id}.log"

        with open(pages_file, "w") as pf:
            json.dump(chunk, pf)
        open(log_file, "w").close()   # clear/create log

        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

        cmd = [
            sys.executable, WORKER_SCRIPT,
            MODEL_ID, pages_file, OUTPUT_DIR, log_file,
            str(DPI), str(MAX_NEW_TOKENS), f"GPU{gpu_id}",
        ]
        p = subprocess.Popen(
            cmd, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        procs.append((gpu_id, p))
        log_files.append((gpu_id, log_file))
        print(f"  → Launched GPU{gpu_id} worker (pid {p.pid})  — {len(chunk)} pages")
        sys.stdout.flush()

    # ── Stream each worker's stdout to notebook output ────────────────────────
    def stream(gpu_id, proc):
        for line in proc.stdout:
            print(f"  [GPU{gpu_id}] {line}", end="", flush=True)

    threads = [
        threading.Thread(target=stream, args=(gid, p), daemon=True)
        for gid, p in procs
    ]
    for t in threads:
        t.start()

    # ── Wait for all workers ──────────────────────────────────────────────────
    for gid, p in procs:
        p.wait()
        ok = p.returncode == 0
        print(f"  GPU{gid} worker {'✓ exited OK' if ok else f'✗ exit code {p.returncode}'}")
        sys.stdout.flush()

    for t in threads:
        t.join()

# ── Summary ───────────────────────────────────────────────────────────────────
total_time = time.time() - overall_start
md_files   = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(".md")]
print(f"\n{'='*60}")
print(f"All workers finished in {fmt(total_time)}")
print(f"Markdown files produced: {len(md_files)}")
print(f"Output directory: {OUTPUT_DIR}")
sys.stdout.flush()


In [ ]:
# Cell 5: Zip Output for Download
import shutil
print("Zipping output files...")
shutil.make_archive("/kaggle/working/output", 'zip', OUTPUT_DIR)
print("✅ Done! file is at /kaggle/working/output.zip")